# 🏢 Real-Time HVAC Monitoring using Spark Structured Streaming

## 📌 Overview

In this project, we build a **real-time streaming data pipeline** using **Apache Spark Structured Streaming** to monitor HVAC conditions in a smart building.

The pipeline simulates continuous sensor readings from multiple rooms and processes them in real time to detect abnormal environmental conditions.

This project demonstrates how Spark can be used for **stream processing**, **real-time analytics**, and **monitoring systems**.

---

## 🎯 Objectives

- Simulate real-time sensor data using Spark's rate source  
- Generate room-level temperature and humidity readings  
- Create a temporary SQL view over streaming data  
- Detect critical temperature conditions in real time  
- Calculate average temperature and humidity over time windows  
- Identify rooms requiring attention based on humidity levels  
- Output streaming results to the console  

---

## 🛠️ Technologies Used

- **Apache Spark**
- **PySpark**
- **Spark Structured Streaming**
- **Spark SQL**
- **Python**

---

## 📡 Simulated Sensor Data

The streaming dataset represents HVAC sensor readings from multiple rooms.

Main fields include:

- `room_id` — unique room identifier  
- `temperature` — simulated temperature reading in °C  
- `humidity` — simulated humidity reading in %  
- `timestamp` — event timestamp generated by Spark  

💡 The data stream simulates continuous sensor readings, similar to real-world IoT monitoring systems.

---

## 💡 Project Structure

The pipeline consists of the following stages:

1. Spark Initialization  
2. Real-Time Data Simulation  
3. Streaming SQL View  
4. Real-Time Analytics Queries  
5. Streaming Output to Console  
6. Streaming Insights  
7. Stop Streaming Queries  

---

💡 This project demonstrates how **data engineering and real-time analytics** can support smart building monitoring and proactive HVAC management.

## ⚙️ 1. Initialize Spark Session

We start by initializing a Spark session, which is the entry point for working with Spark.

We also import the required libraries for:
- Structured Streaming
- SQL transformations
- Data simulation

In [10]:
# Initialize Spark Session (works in most environments)

from pyspark.sql import SparkSession
from pyspark.sql.functions import expr, rand, when

# Initialize Spark Session

spark = SparkSession.builder \
    .appName("Smart Building HVAC Monitoring") \
    .getOrCreate()

## 2. Simulate Real-Time Sensor Data

In this section, we simulate real-time sensor readings using Spark's **rate source**.

The stream generates continuous records, which are transformed into simulated HVAC sensor readings.

Each generated event includes:
- `room_id`
- `temperature`
- `humidity`
- `timestamp`

In [5]:
# Simulate real-time sensor data

sensor_data = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 5) \
    .load() \
    .withColumn("room_id", expr("CAST(value % 10 AS STRING)")) \
    .withColumn(
        "temperature",
        when(expr("value % 10 == 0"), 15)
        .otherwise(20 + rand() * 25)
    ) \
    .withColumn("humidity", expr("40 + rand() * 30"))

print("Streaming DataFrame created successfully")

Streaming DataFrame created successfully


In [6]:
# Inspect streaming schema

sensor_data.printSchema()

root
 |-- timestamp: timestamp (nullable = true)
 |-- value: long (nullable = true)
 |-- room_id: string (nullable = true)
 |-- temperature: double (nullable = false)
 |-- humidity: double (nullable = false)



🔍 Streaming Observations

- A continuous data stream is generated using Spark's rate source
- Each record simulates a real-time HVAC sensor reading
- Temperature includes occasional low values to simulate anomalies
- Humidity values vary within a realistic range
- The streaming dataset mimics real-world IoT sensor behavior

💡 Next step: create a streaming SQL view and run real-time analytics queries.

## 3. Create Streaming SQL View

In this section, we register the streaming DataFrame as a temporary SQL view.

This allows us to run **Spark SQL queries** directly on streaming sensor data.

In [7]:
# Create temporary SQL view for streaming data

sensor_data.createOrReplaceTempView("sensor_table")

print("Temporary SQL view created successfully")

Temporary SQL view created successfully


🔍 Streaming SQL View Observations

- The streaming DataFrame is successfully registered as a temporary SQL view
- This enables running SQL queries directly on real-time data
- The view acts as a bridge between streaming data and analytical queries
- Spark SQL can now be used for real-time data exploration and monitoring

💡 Next step: run real-time analytics queries on the streaming data.

## 4. Define Real-Time Analytics Queries

In this section, we define SQL queries for real-time monitoring and analysis.

The queries are designed to:

- Detect rooms with **critical temperature values**
- Calculate **average temperature and humidity** over a time window
- Identify rooms that require attention based on humidity levels

In [8]:
# Query 1: Detect critical temperature values

critical_temperature_query = """
    SELECT
        room_id,
        temperature,
        humidity,
        timestamp
    FROM sensor_table
    WHERE temperature < 18 OR temperature > 60
"""

In [9]:
# Query 2: Calculate average readings over a 1-minute window

average_readings_query = """
    SELECT
        room_id,
        AVG(temperature) AS avg_temperature,
        AVG(humidity) AS avg_humidity,
        window.start AS window_start
    FROM sensor_table
    GROUP BY room_id, window(timestamp, '1 minute')
"""

In [10]:
# Query 3: Identify rooms requiring attention based on humidity

attention_needed_query = """
    SELECT
        room_id,
        COUNT(*) AS critical_readings
    FROM sensor_table
    WHERE humidity < 45 OR humidity > 75
    GROUP BY room_id
"""

🔍 Real-Time Query Design Summary

- The first query detects critical temperature events in real time
- The second query calculates average temperature and humidity using a 1-minute time window
- The third query identifies rooms that may require attention based on abnormal humidity levels
- These queries represent different real-time monitoring use cases: alerting, aggregation, and operational tracking

💡 Next step: execute these SQL queries as streaming DataFrames.

## 5. Execute Streaming Queries

In this section, we execute the SQL queries and create streaming DataFrames.

Each streaming DataFrame represents a different real-time monitoring output:

- Critical temperature events  
- Average readings by room and time window  
- Humidity-based attention alerts  

In [11]:
# Execute critical temperature query

critical_temperatures_stream = spark.sql(critical_temperature_query)

In [12]:
# Execute average readings query

average_readings_stream = spark.sql(average_readings_query)

In [13]:
# Execute attention-needed query

attention_needed_stream = spark.sql(attention_needed_query)

In [14]:
# Check if DataFrames are streaming

print("Critical temperatures stream:", critical_temperatures_stream.isStreaming)
print("Average readings stream:", average_readings_stream.isStreaming)
print("Attention needed stream:", attention_needed_stream.isStreaming)

Critical temperatures stream: True
Average readings stream: True
Attention needed stream: True


🔍 Streaming Execution Observations

- Each SQL query is converted into a streaming DataFrame
- All resulting DataFrames are confirmed to be streaming (isStreaming = True)
- The pipeline now supports multiple real-time monitoring outputs simultaneously
- This step bridges SQL-based logic with real-time execution in Spark

💡 Next step: write streaming outputs to the console or external sinks.

## 6. Output Streaming Results to Console

In this section, we start streaming queries and output results to the console.

Each query writes live results using a different output mode:

- **Append mode** for critical temperature events  
- **Complete mode** for aggregated average readings  
- **Complete mode** for humidity-based attention alerts  

⚠️ Since streaming queries continuously run, they should be stopped manually after collecting enough sample output.

💡 Note: Spark Structured Streaming displays results in micro-batches. Some batches may be empty if no records match the query conditions during that interval. Internal Spark stage logs may also appear during execution.

In [19]:
# Output critical temperature events to console

critical_query = critical_temperatures_stream.writeStream \
    .outputMode("append") \
    .format("console") \
    .queryName("Critical Temperatures") \
    .start()

-------------------------------------------
Batch: 0
-------------------------------------------
+-------+-----------+--------+---------+
|room_id|temperature|humidity|timestamp|
+-------+-----------+--------+---------+
+-------+-----------+--------+---------+

-------------------------------------------
Batch: 1
-------------------------------------------
+-------+-----------+-----------------+--------------------+
|room_id|temperature|         humidity|           timestamp|
+-------+-----------+-----------------+--------------------+
|      0|       15.0|62.08309087054903|2026-04-30 11:47:...|
+-------+-----------+-----------------+--------------------+

-------------------------------------------
Batch: 2
-------------------------------------------
+-------+-----------+--------+---------+
|room_id|temperature|humidity|timestamp|
+-------+-----------+--------+---------+
+-------+-----------+--------+---------+

-------------------------------------------
Batch: 3
--------------------

In [20]:
# Output average readings to console

average_query = average_readings_stream.writeStream \
    .outputMode("complete") \
    .format("console") \
    .queryName("Average Readings") \
    .start()

-------------------------------------------
Batch: 4
-------------------------------------------
+-------+-----------+--------+---------+
|room_id|temperature|humidity|timestamp|
+-------+-----------+--------+---------+
+-------+-----------+--------+---------+



-------------------------------------------
Batch: 5
-------------------------------------------
+-------+-----------+-----------------+--------------------+
|room_id|temperature|         humidity|           timestamp|
+-------+-----------+-----------------+--------------------+
|      0|       15.0|50.89465142996316|2026-04-30 11:47:...|
+-------+-----------+-----------------+--------------------+

-------------------------------------------
Batch: 0
-------------------------------------------
+-------+---------------+------------+------------+
|room_id|avg_temperature|avg_humidity|window_start|
+-------+---------------+------------+------------+
+-------+---------------+------------+------------+

-------------------------------------------
Batch: 6
-------------------------------------------
+-------+-----------+------------------+--------------------+
|room_id|temperature|          humidity|           timestamp|
+-------+-----------+------------------+--------------------+
|      0

-------------------------------------------
Batch: 7
-------------------------------------------
+-------+-----------+--------+---------+
|room_id|temperature|humidity|timestamp|
+-------+-----------+--------+---------+
+-------+-----------+--------+---------+



[Stage 15:===>           (43 + 9) / 200][Stage 17:>                 (0 + 0) / 8]

In [21]:
# Output humidity attention alerts to console

attention_query = attention_needed_stream.writeStream \
    .outputMode("complete") \
    .format("console") \
    .queryName("Attention Needed") \
    .start()

-------------------------------------------
Batch: 8
-------------------------------------------
+-------+-----------+-----------------+--------------------+
|room_id|temperature|         humidity|           timestamp|
+-------+-----------+-----------------+--------------------+
|      0|       15.0|47.00479816779793|2026-04-30 11:47:...|
|      0|       15.0|49.48935772258888|2026-04-30 11:47:...|
+-------+-----------+-----------------+--------------------+

-------------------------------------------
Batch: 1
-------------------------------------------
+-------+------------------+------------------+-------------------+
|room_id|   avg_temperature|      avg_humidity|       window_start|
+-------+------------------+------------------+-------------------+
|      2|31.026479445412978| 57.45245246611103|2026-04-30 11:47:00|
|      9| 29.90318110103401| 57.80787360388554|2026-04-30 11:47:00|
|      5| 36.67429668845038| 50.58827615502632|2026-04-30 11:47:00|
|      8| 34.07220961044989| 55

[Stage 19:=>             (20 + 8) / 200][Stage 20:>                 (0 + 0) / 8]

🔍 Streaming Output Observations

- Streaming queries run continuously and output results in micro-batches
- Some batches may appear empty if no records match query conditions
- Spark execution logs and stage progress may appear in the console during execution
- These logs are part of normal streaming behavior and not errors
- Different output modes are used depending on query type (append vs complete)


## 7. Stop Streaming Queries

Streaming queries are designed to run continuously in real-time systems.

For this demonstration, we manually stop all active queries after capturing representative output samples.

💡 This approach ensures controlled execution while still demonstrating real-time data processing capabilities.

In [24]:
# Stop all active streaming queries

for query in spark.streams.active:
    print("Stopping:", query.name)
    query.stop()

print("All streaming queries stopped")

All streaming queries stopped


### 8. Stop Spark Session

In the final step, we stop the Spark session to release system resources.

This is an important best practice when working with Spark and streaming applications.

In [25]:
# Stop Spark session

spark.stop()

print("Spark session stopped")

Spark session stopped


---

## 9. Streaming Insights

Although streaming output is continuous, we can interpret the system behavior based on the defined queries and simulated data patterns.

### Key Observations

- The system continuously processes incoming sensor data in real time (~5 records per second)
- Rooms with **critical temperature values** (e.g., 15°C) are immediately detected
- **Humidity anomalies** trigger alerts for specific rooms
- Rolling averages provide insights into environmental trends over time

### 💡 Interpretation

- Real-time monitoring enables **instant detection of abnormal conditions**
- The system can identify rooms that require **immediate maintenance or inspection**
- Aggregated metrics support **long-term optimization of HVAC performance**

⚡ This demonstrates how streaming pipelines enable **proactive system monitoring instead of reactive responses**.

## 10. Why This Matters

Streaming data processing is essential in modern systems where:

- Data is generated continuously (IoT, sensors, logs)
- Immediate decisions are required
- Systems must react in real time

### 🚀 Benefits of Using Spark Streaming

- Handles **high-velocity data streams**
- Provides **scalable distributed processing**
- Supports **real-time analytics with SQL**
- Enables **low-latency monitoring systems**

💡 This makes Spark a powerful tool for building real-time data engineering pipelines.

## 11. Conclusion

This project demonstrates how Apache Spark Structured Streaming can be used to process and analyze real-time sensor data in smart building environments.

Using Spark, we successfully:

- Simulated continuous HVAC sensor data streams  
- Built a real-time streaming data pipeline  
- Applied SQL-based analytics on live data  
- Detected critical environmental conditions  
- Generated actionable monitoring insights  

---

### 🚀 Key Value

- Enables **real-time decision-making**
- Improves **system reliability and comfort**
- Supports **predictive maintenance**
- Scales across large sensor networks  

💡 This project highlights the importance of real-time data engineering in modern smart systems and IoT applications.